In [ ]:
import numpy as np
from learn_s_hat_toy import make_srs
import copy
from adaptive_latents.utils import save_to_cache
from adaptive_latents import ArrayWithTime
import matplotlib.pyplot as plt

In [ ]:
output = None
use_cache = True

In [ ]:
@save_to_cache('spinning_toy')
def f(u_function='curvy spins', n_runs=100):
    rng = np.random.default_rng(0)
    srs = make_srs(copy.deepcopy(rng), n_runs=n_runs, u_function=u_function, show_tqdm=True, n_rotations=100)
    return rng, srs

rng, srs = f('curvy spins', n_runs=50, _recalculate_cache_value=not use_cache)


In [ ]:
%matplotlib inline
def square_err_array(errs):
    lengths = [len(e.t) for e in errs]
    common_length = min(lengths)
    longest_idx = np.argmax(lengths)
    # a = [np.linalg.norm(e.slice(slice(-common_length, None))**2, axis=1) for e in errs]
    # t = errs[0].slice(slice(-common_length, None)).t
    # return ArrayWithTime(a, t)

    max_length = max(lengths)

    assert np.std(np.vstack([e.t[-common_length:] for e in errs]), axis=0).max() < 1e-10

    filled_errors = []
    for e in errs:
        early_nans = np.array([e[0] * np.nan] * (max_length - len(e))).reshape(-1,3)
        padded_e = np.vstack([np.squeeze(early_nans), e])
        filled_errors.append(padded_e)


    a = np.squeeze([np.linalg.norm(e, axis=1) for e in filled_errors])
    t = errs[longest_idx].t
    good_idx = np.nonzero(np.isnan(a).sum(axis=0) < int(a.shape[0] * .75))[0][0]
    good_idx = 10
    ret = ArrayWithTime(a[:,good_idx:], t[good_idx:])
    return ret


fig, ax = plt.subplots(figsize=np.array((56,23.6))/5, layout='constrained')

ax2 = ax.twinx()

kernel = np.hstack([np.zeros(50), np.ones(51)])
# kernel = np.ones(51)
kernel = kernel / kernel.sum()
time_kernel = kernel * 0
time_kernel[len(kernel)//2] = 1

errs = square_err_array([sr.log['pred_error'] for sr in srs['unaware of stim']])
mean_errors = np.nanmean(errs, axis=0)
ax.plot(errs.t, errs[0], label='unaware (single trial)', color='#4d4d4dff', alpha=.1,)
smoothed_mean_errs = ArrayWithTime(np.convolve(mean_errors, kernel, 'valid'), np.convolve(errs.t, time_kernel, 'valid'))
ax2.plot(smoothed_mean_errs.t, smoothed_mean_errs, label='unaware (averaged, smoothed)', color='#4d4d4dff')

errs = square_err_array([sr.log['pred_error'] for sr in srs['learning from stim']])
mean_errors = np.nanmean(errs, axis=0)
ax.plot(errs.t, errs[0], label='aware (single trial)', color='#ca1469ff', alpha=.1)
smoothed_mean_errs = ArrayWithTime(np.convolve(mean_errors, kernel, 'valid'), np.convolve(errs.t, time_kernel, 'valid'))
ax2.plot(smoothed_mean_errs.t, smoothed_mean_errs, label='aware (averaged, smoothed)', color='#ca1469ff')

ax2.axvline(25, color='gray', linestyle='--')
ax2.axvline(45, color='gray', linestyle='--')
ax2.axvline(75, color='gray', linestyle='--')

ax.set_xlabel('time (s)')
ax.set_ylabel('error')
ax2.set_ylabel('averaged error')
ax2.legend(loc='upper right')
ax.legend(loc='upper left')
ax.set_xlim([0,100])
ax2.set_xlim([0,100])

# ax.set_ylim([0,290])
# ax2.set_ylim([0,13])

# x = np.linspace(0,50,200)
# ax2.plot(x+50,  % (2 * np.pi))

if output is not None:
    fig.savefig(output)


In [ ]:
fig, ax = plt.subplots()
for sr in srs['learning from stim']:
    sr.stim_reg.plot_length_scales(ax)